# Tổng hợp các thí nghiệm HeteroES (đêm 24 → 25/11/2026)

Bảng gom kết quả các notebook thí nghiệm (`results/*.json`). "Đúng" = điểm 1 trên tập test, chấm **một lần** ở cuối;
p là kiểm định McNemar chính xác (mô hình gốc so với checkpoint được chọn). Run Countdown × Qwen2.5-0.5B có từ trước
(test 0 → 32/249) nằm trong báo cáo, không lặp lại ở đây.

In [1]:
import json
from pathlib import Path

HERE = Path.cwd()
print(f"{'mô hình':30} {'thí nghiệm':22} {'n':>5} {'gốc':>5} {'sau ES':>7} {'p':>9}  thế hệ")
for path in sorted((HERE / "results").glob("*.json")):
    s = json.loads(path.read_text())
    t = s["test"]
    state = json.loads((HERE / "state" / path.name).read_text())["segments"]
    planned = sum(x["generations"] for x in s["segments"])
    early = [v["endedEarly"] for v in state.values() if v.get("endedEarly")]
    note = f"{planned} (dừng sớm: {early[0]})" if early else str(planned)
    print(f"{s['model']:30} {s['name']:22} {t['base']['count']:5} {t['base']['correct']:5} "
          f"{t['selected']['correct']:7} {s['comparison']['mcnemarExactP']:9.2g}  {note}")

mô hình                        thí nghiệm                 n   gốc  sau ES         p  thế hệ
Qwen/Qwen2.5-0.5B-Instruct     e1-gsm8k-qwen05         1319   462     528   7.1e-05  100
HuggingFaceTB/SmolLM2-360M-Instruct e2-arith-smollm2         300     0       0         1  120
Qwen/Qwen2.5-1.5B-Instruct     e3-gsm8k-qwen15         1319   703     871   9.2e-21  100
Qwen/Qwen2.5-1.5B-Instruct     e4-countdown-qwen15      250     4      31   1.1e-07  150 (dừng sớm: lab hủy run ở thế hệ 123/150 lúc 10:33 ngày 25/11 do lỗi lab (gỡ một worker rồi tắt máy đó làm hủy cả run; đã sửa ở commit 85e3d630). Dùng điểm khôi phục đã kiểm chứng mới nhất: thế hệ 110.)


## Tốc độ trong run thật: 1 GPU so với 2 GPU

Đọc từ ledger của coordinator (máy 5070 Ti của Nhân), nơi ghi **mọi** candidate: máy nào chấm, lúc nào xong.
Thời gian một thế hệ = khoảng cách giữa hai lần ghi cập nhật trọng số (gồm cả các lần chấm validation mỗi 25 thế hệ,
nên dùng trung vị). Audit = một máy khác chấm lại một candidate; phải ra cùng SHA-256 của câu trả lời.

In [2]:
import collections
import sqlite3
import statistics

RUNS = Path.home() / ".local/share/heteroes-worker/agent-state/runs"
NAMES = {"cli-3dacd1745773": "5070ti-nhan", "cli-d152aa6a13f8": "5070ti-nhut", "cli-1a9e23ce56d1": "3060-quang",
         "cli-a6c1c55a80d8": "3060-thao"}


def speed(run_id):
    db = sqlite3.connect(f"file:{RUNS / run_id / 'ledger.sqlite'}?mode=ro", uri=True)
    done = dict(db.execute("SELECT generation, recorded_at FROM generation_update"))
    who = collections.defaultdict(set)
    for g, w in db.execute(
        "SELECT c.generation, a.worker_id FROM result r JOIN attempt a ON a.candidate_id = r.candidate_id "
        "AND a.attempt_number = r.attempt_number JOIN candidate c ON c.candidate_id = r.candidate_id"
    ):
        who[g].add(NAMES.get(w, w))
    groups = collections.defaultdict(list)
    for g in sorted(done):
        if g - 1 in done and who.get(g):
            groups[" + ".join(sorted(who[g]))].append(done[g] - done[g - 1])
    audits, matched = db.execute("SELECT COUNT(*), COALESCE(SUM(matched), 0) FROM audit").fetchone()
    return groups, audits, matched


for name in ("e3-gsm8k-qwen15", "e4-countdown-qwen15"):
    run_id = json.loads((HERE / "state" / f"{name}.json").read_text())["segments"]["segment-01"]["experimentId"]
    if not (RUNS / run_id / "ledger.sqlite").is_file():
        print(name, ": ledger không có trên máy này")
        continue
    groups, audits, matched = speed(run_id)
    print(f"{name}: audit khớp {matched}/{audits}")
    for machines, seconds in sorted(groups.items(), key=lambda kv: -len(kv[1])):
        print(f"   {machines:28} {len(seconds):4} thế hệ, trung vị {statistics.median(seconds):6.1f} s/thế hệ")

e3-gsm8k-qwen15: audit khớp 5/5
   5070ti-nhan                    97 thế hệ, trung vị  169.0 s/thế hệ
   3060-quang + 5070ti-nhan        2 thế hệ, trung vị  150.9 s/thế hệ
e4-countdown-qwen15: audit khớp 190/190
   5070ti-nhan + 5070ti-nhut     118 thế hệ, trung vị   80.5 s/thế hệ
   5070ti-nhan                     4 thế hệ, trung vị  169.9 s/thế hệ


Đọc kết quả: E4 chạy 4 thế hệ trên một 5070 Ti rồi máy 5070 Ti thứ hai (của Nhựt) được thêm vào **giữa run**
(không dừng run). Mốc một máy chỉ có 4 thế hệ, gồm cả lúc nạp mô hình, nên tỷ lệ tăng tốc chỉ là ước lượng.
Máy 3060 của Thảo được thêm vào E4 nhưng không gửi được kết quả về coordinator (mạng WSL rớt gói lớn), nên không có
trong bảng; máy 3060 của Quang chỉ tham gia 2 thế hệ đầu của E3 trước khi ổ C: đầy.